# Adult Income Prediction and Algorithmic Bias Analysis

**Objective:** Predict whether an individual's annual income exceeds $50,000 using the UCI Adult Income census dataset. In addition to optimizing predictive performance, this notebook focuses on detecting and mitigating algorithmic bias related to two sensitive attributes: `sex` and `race`, using the [`fairlearn`](https://fairlearn.org/) library.

**Notebook structure:**

1. Load and clean the data
2. Build the preprocessing pipeline
3. Cross validation and hyperparameter tuning with `RandomizedSearchCV`
4. Evaluate the tuned baseline model on the test set
5. Measure fairness with `fairlearn.metrics.MetricFrame` for sex, race, and their intersection
6. Mitigate bias with `fairlearn.reductions.ExponentiatedGradient`
7. Compare results before and after mitigation and analyze the trade off between Demographic Parity and Equalized Odds
8. Interpret the model with SHAP and identify features that may act as proxies for sex or race
9. Conclusions and project limitations

## 1. Load and Clean the Data

The original dataset uses the character `"?"` to represent missing values in the `workclass`, `occupation`, and `native.country` columns. These values are replaced with `NaN`, and rows containing missing values are removed.

The target label `income_label` is set to 1 when income is `>50K` and 0 otherwise.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

from fairlearn.metrics import (
    MetricFrame,
    selection_rate,
    true_positive_rate,
    false_positive_rate,
    demographic_parity_difference,
    equalized_odds_difference,
)
from fairlearn.reductions import ExponentiatedGradient, DemographicParity

import shap

RANDOM_STATE = 42
pd.set_option("display.float_format", lambda v: f"{v:0.4f}")

In [ ]:
df = pd.read_csv("adult_income.csv")
df = df.replace("?", np.nan)
n_before = len(df)
df = df.dropna()
n_after = len(df)
print(f"Rows before and after removing missing values: {n_before} -> {n_after} ({n_before - n_after} rows removed, "
      f"{(n_before - n_after) / n_before:.1%})")

df["income_label"] = (df["income"] == ">50K").astype(int)
df["income_label"].value_counts(normalize=True).rename("proportion").to_frame()

**Note about `fnlwgt`:** This column represents the Census Bureau sampling weight and does not provide individual level predictive information. It is therefore excluded from the feature set, consistent with the original approach.

**Feature set design:** `sex` and `race` are retained separately as sensitive attributes for fairness evaluation. They are not included in the model input `X`. This approach is known as *fairness through unawareness*, which directly excludes sensitive attributes from the model inputs. Section 8 uses SHAP to examine whether this approach actually removes bias.

In [ ]:
numeric_features = ["age", "education.num", "capital.gain", "capital.loss", "hours.per.week"]
categorical_features = ["workclass", "education", "marital.status", "occupation", "relationship"]
feature_cols = numeric_features + categorical_features

X = df[feature_cols].copy()
y = df["income_label"].copy()
sensitive = df[["sex", "race"]].copy()

X_train, X_test, y_train, y_test, A_train, A_test = train_test_split(
    X, y, sensitive, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

# Combine sex and race to analyze intersectional fairness in Section 5
A_train["sex_race"] = A_train["sex"] + " & " + A_train["race"]
A_test["sex_race"] = A_test["sex"] + " & " + A_test["race"]

print("Training set:", X_train.shape, " Test set:", X_test.shape)

## 2. Preprocessing Pipeline

Instead of applying `LabelEncoder`, which can create an artificial order among categorical values, to the full dataset before splitting the data, this notebook uses `ColumnTransformer` and `OneHotEncoder` inside a `sklearn.Pipeline`. The pipeline is fitted only on `X_train`, preventing information from the test set from leaking into preprocessing. It can also handle previously unseen categories through `handle_unknown="ignore"` when used in practice.

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ]
)
print("Preprocessor is ready.")

## 3. Cross Validation and Hyperparameter Tuning

The original version used fixed parameters (`n_estimators=200, max_depth=12`) without a clear selection process. This notebook uses `RandomizedSearchCV` with `StratifiedKFold` and 3 folds to preserve class proportions in each fold. The best parameter set is selected using the mean AUC across folds. AUC is more suitable than accuracy alone when the positive class represents only about 25% of the data.

The search space and number of iterations are kept moderate so that the notebook can run within a few minutes. With more computing resources or time, `param_distributions`, `n_iter`, and `cv` can be expanded.

In [ ]:
tuning_pipeline = Pipeline(steps=[
    ("prep", preprocessor),
    ("clf", RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)),
])

param_distributions = {
    "clf__n_estimators": [100, 150, 200],
    "clf__max_depth": [8, 12, 16],
    "clf__min_samples_leaf": [1, 2, 4],
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

search = RandomizedSearchCV(
    tuning_pipeline,
    param_distributions=param_distributions,
    n_iter=6,
    cv=cv,
    scoring="roc_auc",
    random_state=RANDOM_STATE,
    n_jobs=1,
    verbose=0,
)
search.fit(X_train, y_train)

print("Best params:", search.best_params_)
print(f"Best CV AUC: {search.best_score_:.4f}")

cv_results = pd.DataFrame(search.cv_results_)[
    ["params", "mean_test_score", "std_test_score", "rank_test_score"]
].sort_values("rank_test_score")
cv_results

## 4. Evaluate the Tuned Baseline Model on the Test Set

In [ ]:
baseline_pipeline = search.best_estimator_

y_pred = baseline_pipeline.predict(X_test)
y_proba = baseline_pipeline.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_proba)

print(f"Accuracy: {acc:.4f}")
print(f"AUC:      {auc:.4f}\n")
print(classification_report(y_test, y_pred, target_names=["<=50K", ">50K"]))

## 5. Measure Fairness with `fairlearn`

This notebook uses three fairness measures.

**Selection rate:** The proportion of individuals predicted to have income above $50,000 in each group.

**Demographic Parity difference (DP diff):** The largest difference in selection rates across groups. A value of zero means the model predicts income above $50,000 at the same rate for every group, regardless of the true labels.

**Equalized Odds difference (EO diff):** The largest difference across groups in both the True Positive Rate (TPR, also called Equal Opportunity) and the False Positive Rate (FPR).

In addition to evaluating each sensitive attribute separately, this notebook examines the intersection of `sex` and `race`. A group may appear fair when each attribute is considered separately but experience a larger disadvantage when both attributes are considered together. For example, Black women may experience a different outcome from the overall groups of women or Black individuals.

In [ ]:
def fairness_table(y_true, y_pred, sensitive_col, label):
    mf = MetricFrame(
        metrics={
            "selection_rate": selection_rate,
            "tpr (recall)": true_positive_rate,
            "fpr": false_positive_rate,
        },
        y_true=y_true,
        y_pred=y_pred,
        sensitive_features=sensitive_col,
    )
    print(f"=== Fairness by '{label}' ===")
    display(mf.by_group)
    dp = demographic_parity_difference(y_true, y_pred, sensitive_features=sensitive_col)
    eod = equalized_odds_difference(y_true, y_pred, sensitive_features=sensitive_col)
    print(f"Demographic Parity diff : {dp:.4f}")
    print(f"Equalized Odds diff     : {eod:.4f}\n")
    return dp, eod

dp_sex_base, eod_sex_base = fairness_table(y_test, y_pred, A_test["sex"], "sex")
dp_race_base, eod_race_base = fairness_table(y_test, y_pred, A_test["race"], "race")

In [ ]:
# Intersectional fairness: sex x race. Some groups have very few samples, such as Amer-Indian-Eskimo Female,
# so their selection_rate, TPR, and FPR may be unstable. Interpret results with sample size in mind.
group_sizes = A_test["sex_race"].value_counts()
print("Sample count for each intersectional group (sex x race):")
display(group_sizes.to_frame("sample_count"))

dp_inter, eod_inter = fairness_table(y_test, y_pred, A_test["sex_race"], "sex x race (intersection)")

**Baseline results:** Although `sex` and `race` are not included as model features, fairness disparities remain visible for both attributes. The DP and EO differences for intersectional groups are larger than those for some individual attributes, suggesting that the smallest intersectional groups may experience combined disadvantages. This supports the conclusion that removing sensitive attributes from model inputs through *fairness through unawareness* is not sufficient to eliminate bias. The model may still learn correlated signals through proxy features, as discussed in Section 8.

Some intersectional groups have very few samples in the test set. Their TPR and FPR estimates may therefore have high variance and should be interpreted with caution rather than treated as definitive estimates.

## 6. Mitigate Bias with `fairlearn.reductions.ExponentiatedGradient`

Rather than manually implementing a reweighting formula, this notebook uses the **Exponentiated Gradient** algorithm from `fairlearn`. This in processing technique repeatedly trains multiple classifiers with different sample weights and combines them into a randomized classifier that aims to satisfy a specified fairness constraint. Here, the `DemographicParity()` constraint is applied to `sex`, using the tuned Random Forest parameters selected in Section 3.

The constraint is intentionally optimized only for `sex`. Fairness is then measured again for `sex`, `race`, and their intersection to examine whether mitigating bias for one attribute has positive or negative spillover effects on the other attributes and groups.

In [ ]:
best_params = search.best_params_

mitigator_estimator = Pipeline(steps=[
    ("prep", preprocessor),
    ("clf", RandomForestClassifier(
        n_estimators=best_params["clf__n_estimators"],
        max_depth=best_params["clf__max_depth"],
        min_samples_leaf=best_params["clf__min_samples_leaf"],
        random_state=RANDOM_STATE, n_jobs=-1,
    )),
])

mitigator = ExponentiatedGradient(
    estimator=mitigator_estimator,
    constraints=DemographicParity(),
    sample_weight_name="clf__sample_weight",
    max_iter=30,
)

mitigator.fit(X_train, y_train, sensitive_features=A_train["sex"])
y_pred_mit = mitigator.predict(X_test)

acc_mit = accuracy_score(y_test, y_pred_mit)
print(f"Accuracy after mitigation: {acc_mit:.4f}  (baseline: {acc:.4f}, difference {acc_mit - acc:+.4f})")

In [ ]:
dp_sex_mit, eod_sex_mit = fairness_table(y_test, y_pred_mit, A_test["sex"], "sex (after mitigation)")
dp_race_mit, eod_race_mit = fairness_table(y_test, y_pred_mit, A_test["race"], "race (after mitigation)")
dp_inter_mit, eod_inter_mit = fairness_table(y_test, y_pred_mit, A_test["sex_race"], "sex x race (after mitigation)")

## 7. Compare Results Before and After Mitigation and Analyze Trade Offs

In [ ]:
summary = pd.DataFrame({
    "Accuracy": [acc, acc_mit],
    "DP diff (sex)": [dp_sex_base, dp_sex_mit],
    "EO diff (sex)": [eod_sex_base, eod_sex_mit],
    "DP diff (race)": [dp_race_base, dp_race_mit],
    "EO diff (race)": [eod_race_base, eod_race_mit],
    "DP diff (sex x race)": [dp_inter, dp_inter_mit],
    "EO diff (sex x race)": [eod_inter, eod_inter_mit],
}, index=["Baseline (tuned)", "After mitigation (ExponentiatedGradient, DP by sex)"])
summary

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

labels = ["DP diff", "EO diff"]
x = np.arange(len(labels))
w = 0.35

panels = [
    ("SEX (constraint optimized)", [dp_sex_base, eod_sex_base], [dp_sex_mit, eod_sex_mit]),
    ("RACE (not directly optimized)", [dp_race_base, eod_race_base], [dp_race_mit, eod_race_mit]),
    ("SEX x RACE (intersection)", [dp_inter, eod_inter], [dp_inter_mit, eod_inter_mit]),
]

for ax, (title, before, after) in zip(axes, panels):
    ax.bar(x - w/2, before, w, label="Before", color="tomato")
    ax.bar(x + w/2, after, w, label="Sau mitigation", color="seagreen")
    ax.set_xticks(x); ax.set_xticklabels(labels)
    ax.set_title(title)
    ax.set_ylabel("Difference")
    ax.legend()

plt.tight_layout()
plt.savefig("fairness_before_after.png", dpi=120)
plt.show()

### Why Does DP Difference Decrease While EO Difference Increases?

The table and chart above illustrate an important issue in fairness machine learning. When the `DemographicParity` constraint is optimized for `sex`, the DP difference for sex decreases substantially. The model predicts income above $50,000 at more similar rates for men and women. However, the EO difference for sex increases. This is not necessarily an implementation error. It can occur because the two groups have different base rates, meaning that the actual proportions earning above $50,000 are not equal in the dataset.

**Intuition:** To raise the selection rate of a group with a lower base rate, such as women, until it matches the rate of a group with a higher base rate, such as men, the model may predict income above $50,000 more readily for women. This can increase both TPR and FPR for women in a way that differs from the changes for men. As a result, the difference in TPR and FPR between the groups can widen even while the difference in selection rates becomes smaller.

This illustrates a widely discussed theoretical result in fairness research (Kleinberg et al., 2016; Chouldechova, 2017): when base rates differ across groups, Demographic Parity and Equalized Odds generally cannot both be satisfied at the same time, except in certain special cases. System designers must choose the fairness definition that best fits the application rather than assuming that one metric is universally correct.

**Practical implications for recruitment or income assessment:**

1. If a system is used to screen applications for loans or benefits based on estimated income, prioritizing **Demographic Parity** can help ensure similar approval rates across demographic groups.
2. If the system is used to verify eligibility, **Equal Opportunity or Equalized Odds** may be more important because the goal is to avoid overlooking eligible individuals in any group.
3. Improving one fairness metric may worsen another metric and reduce accuracy. These trade offs should be reported transparently to decision makers rather than reporting only the metric that improved.

**Spillover effects on `race` and intersectional groups:** Although the constraint is optimized only for `sex`, the results for `race` and the `sex x race` intersection also change. This suggests that these attributes are correlated in the dataset through features such as occupation and working hours. Intervening on one attribute can therefore affect other attributes or intersections in positive or negative ways. Real world fairness systems should monitor multiple sensitive attributes and their intersections rather than optimizing only one attribute.

## 8. Model Interpretability with SHAP and Proxy Features for Sex and Race

Although `sex` and `race` are excluded from the model inputs, bias remains, as shown in Section 5. The key question is which of the 11 model features may act as the strongest proxies by indirectly carrying information about sex or race.

`shap.TreeExplainer` is used to calculate SHAP values for the tuned baseline model. The analysis then follows two steps.

1. Rank features by their mean absolute SHAP values to measure global feature importance.
2. For the highest ranked features, examine their distributions across `sex` and `race` using cross tabulations. Features that are important to the model and strongly distributed differently across these groups may be potential proxy features.

To reduce runtime, SHAP values are calculated on a random sample of up to 500 test records rather than the entire test set. This sample provides an estimate of the ranking of average feature importance.

In [ ]:
rng = np.random.RandomState(RANDOM_STATE)
sample_idx = rng.choice(X_test.index, size=min(500, len(X_test)), replace=False)
X_shap = X_test.loc[sample_idx]

fitted_preprocessor = baseline_pipeline.named_steps["prep"]
X_shap_encoded = fitted_preprocessor.transform(X_shap)
if hasattr(X_shap_encoded, "toarray"):
    X_shap_encoded = X_shap_encoded.toarray()

feature_names = numeric_features + list(
    fitted_preprocessor.named_transformers_["cat"].get_feature_names_out(categorical_features)
)

rf_model = baseline_pipeline.named_steps["clf"]
explainer = shap.TreeExplainer(rf_model)
shap_values = explainer.shap_values(X_shap_encoded)

# In recent SHAP versions, shap_values has shape (n_samples, n_features, n_classes); select class ">50K" (index 1)
if isinstance(shap_values, list):
    sv_pos = shap_values[1]
else:
    sv_pos = shap_values[:, :, 1] if shap_values.ndim == 3 else shap_values

mean_abs_shap = pd.Series(np.abs(sv_pos).mean(axis=0), index=feature_names).sort_values(ascending=False)
print("Top 10 most important features by SHAP (mean absolute value):")
mean_abs_shap.head(10)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
top10 = mean_abs_shap.head(10).sort_values()
ax.barh(top10.index, top10.values, color="steelblue")
ax.set_xlabel("Mean |SHAP value|")
ax.set_title("Top 10 Most Important Features (SHAP)")
plt.tight_layout()
plt.savefig("shap_top_features.png", dpi=120)
plt.show()

In [ ]:
# Check which original features, before one-hot encoding, appear in the top 10 SHAP features,
# and examine whether their distributions differ substantially by sex and race, making them potential proxies.
top_raw_features = sorted({f.split("_")[0] for f in mean_abs_shap.head(10).index if f in numeric_features}
                           | {c for c in categorical_features
                              for f in mean_abs_shap.head(10).index if f.startswith(c + "_")})
print("Original features represented in the top 10 SHAP features:", top_raw_features)

for feat in [f for f in top_raw_features if f in categorical_features]:
    print(f"\n--- Distribution of '{feat}' by sex (%) ---")
    display((pd.crosstab(df[feat], df["sex"], normalize="columns") * 100).round(1))
    print(f"--- Distribution of '{feat}' by race (%) ---")
    display((pd.crosstab(df[feat], df["race"], normalize="columns") * 100).round(1))

**Interpretation:** Features such as `relationship` (Husband, Wife, Own child, and others) and `marital.status` may rank among the most important model features according to SHAP while also having distributions that differ substantially by `sex`. For example, the value `Husband` appears almost exclusively among men in the 1994 Census dataset. These features may act as **proxy features**, helping explain why excluding `sex` and `race` from the input does not eliminate bias. The model may learn indirect signals through highly correlated attributes.

Simply removing `sex` and `race` from the input may therefore be insufficient to ensure fairness. Further approaches could include techniques such as `CorrelationRemover` from `fairlearn` or combining or removing categorical levels with highly uneven distributions.

## 9. Conclusions and Project Limitations

**Conclusions**

1. The preprocessing pipeline using `ColumnTransformer` and `OneHotEncoder` inside a `Pipeline` helps prevent preprocessing data leakage compared with the original approach. `RandomizedSearchCV` and `StratifiedKFold` select model parameters based on mean AUC across multiple folds rather than relying on fixed parameters.
2. Excluding `sex` and `race` from the feature set is not sufficient to eliminate bias. SHAP indicates that bias may be reflected through proxy features such as `relationship` and `marital.status`, which are correlated with sex and race.
3. The intersectional analysis of `sex x race` shows that fairness disparities for some small groups can be larger than those observed when each attribute is examined separately. Sufficient sample sizes are needed for reliable conclusions.
4. `ExponentiatedGradient`, with a `DemographicParity` constraint on `sex`, substantially reduces the DP difference for sex, but the EO difference for sex increases and accuracy decreases slightly. This demonstrates the trade off between fairness definitions when groups have different base rates.
5. Fairness interventions targeting one attribute may have unintended spillover effects on other correlated sensitive attributes and intersections.

**Limitations**

1. The 1994 Census dataset is outdated, and current income and employment distributions may differ considerably.
2. The hyperparameter search space is limited to keep runtime short. A production implementation should expand `param_distributions`, increase `n_iter`, and use more cross validation folds.
3. SHAP values are calculated on a sample of 500 records to reduce runtime rather than on the entire test set.
4. Some `sex x race` intersectional groups have very few test samples. Their TPR and FPR estimates may have high variance. More data or statistical methods such as bootstrap confidence intervals would be needed for stronger conclusions.